# Notebook 6: Outlier Treatment


In [1]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    "Transaction_ID": [101, 102, 103, 104, 105, 106, 107, 108],
    "Amount": [900, 1000, 1100, 1200, 1300, 1400, 1500, 20000],
    "Verified": [True, True, True, True, True, True, True, True]
})
print(sales)

   Transaction_ID  Amount  Verified
0             101     900      True
1             102    1000      True
2             103    1100      True
3             104    1200      True
4             105    1300      True
5             106    1400      True
6             107    1500      True
7             108   20000      True


## 1. What is an Outlier?

### Definition
An outlier is a value noticeably different from most observations. It can be unusually high or low. An outlier is a signal for investigation, not automatic deletion. Its importance depends on the dataset and business context.

### Business Example
Most store purchases are around ₹900–₹1,500, but one purchase is ₹20,000. The larger purchase stands apart from the others. It might be a genuine bulk order. The store should inspect its transaction details before changing it.

### AI/ML Use
Extreme values can affect averages and model training. Some models, such as linear regression, are sensitive to them. Other models may handle them better. Identifying outliers helps choose suitable preprocessing.

### Python Example

In [3]:
print("Typical amounts:", sales["Amount"].median())
print("Largest amount:", sales["Amount"].max())

Typical amounts: 1250.0
Largest amount: 20000


### Explanation
The median shows a typical transaction amount. The maximum reveals a much larger value. This difference suggests a possible outlier. It does not prove the transaction is wrong.

## 2. Outlier vs Error

### Definition
An outlier is statistically unusual; an error is factually incorrect. A valid extreme observation is still real data. An error may result from a typo, faulty sensor, or wrong unit. Both should be checked before treatment.

### Business Example
A ₹20,000 verified purchase is unusual but valid. An age of −5 is impossible and indicates an error. The two cases require different decisions. Business records help determine which values are trustworthy.

### AI/ML Use
Deleting valid extremes may remove fraud patterns or important customers. Keeping errors can distort model predictions. Data checks should combine statistics with domain rules. Record the reason for any correction or deletion.

### Python Example

In [5]:
ages = pd.Series([22, 25, 29, -5])
print("Invalid ages:", ages[ages < 0].tolist())
print("Verified large purchase:", sales.loc[sales["Amount"].idxmax(), "Verified"])

Invalid ages: [-5]
Verified large purchase: True


### Explanation
A negative age violates a basic validity rule. The verified flag supports treating the large purchase as genuine. Statistical unusualness and factual correctness are separate questions. Verify records before choosing a treatment.

## 3. IQR Method

### Definition
The interquartile range (IQR) measures the spread of the middle 50% of values. Q1 is the 25th percentile and Q3 is the 75th percentile. IQR equals Q3 minus Q1. Values beyond 1.5 × IQR below Q1 or above Q3 are commonly flagged.

### Business Example
A retailer uses IQR to flag unusually large orders. The method is useful when sales are skewed. Flagged orders go to a review list. Verified bulk orders remain in the records.

### AI/ML Use
IQR does not assume a normal distribution. It is useful for exploratory screening of numerical features. Small datasets may produce unstable boundaries. The flags must still be checked against business rules.

### Python Example

In [7]:
q1 = sales["Amount"].quantile(0.25)
q3 = sales["Amount"].quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(sales[(sales["Amount"] < low) | (sales["Amount"] > high)])

   Transaction_ID  Amount  Verified
7             108   20000      True


### Explanation
The code calculates quartiles and the IQR boundaries. It then displays amounts outside those limits. These are potential outliers, not confirmed errors. Each flagged transaction still needs review.

## 4. Z-Score Method

### Definition
A z-score measures how far a value lies from the mean in standard deviation units. Large absolute z-scores indicate unusual values. A threshold of 3 is common but not universal. The method is most meaningful for roughly symmetric data.

### Business Example
A factory tracks machine measurements near a stable average. A reading several standard deviations away may indicate a fault. Engineers check the equipment before deleting the reading. A real malfunction is useful evidence.

### AI/ML Use
Z-scores can detect unusual measurements in near-normal features. Extreme values themselves affect the mean and standard deviation. Skewed data can make the method misleading. Consider robust methods when distributions are highly skewed.

### Python Example

In [9]:
measurements = pd.Series([49, 50, 51, 50, 49, 52, 50, 90])
z = (measurements - measurements.mean()) / measurements.std(ddof=0)
print(pd.DataFrame({"value": measurements, "z_score": z.round(2)}))

   value  z_score
0     49    -0.46
1     50    -0.39
2     51    -0.31
3     50    -0.39
4     49    -0.46
5     52    -0.24
6     50    -0.39
7     90     2.64


### Explanation
Each measurement is standardized using the sample values. Larger absolute scores indicate greater distance from the mean. A high score is a review flag rather than a deletion command. The distribution and domain rules still matter.

## 5. Percentile Method

### Definition
The percentile method flags values outside chosen percentile limits. For example, values below the 5th or above the 95th percentile can be reviewed. The limits are data-dependent and easy to interpret. The chosen cutoffs should reflect the use case.

### Business Example
A delivery company checks the slowest 5% of deliveries. These cases may represent traffic problems or genuine long-distance trips. They should not automatically disappear from reports. The team first checks the cause.

### AI/ML Use
Percentiles work with skewed features and do not require normality. They can identify tails for inspection or capping. A fixed percentile always selects some observations even in clean data. Do not assume every flagged value is wrong.

### Python Example

In [11]:
p5 = sales["Amount"].quantile(0.05)
p95 = sales["Amount"].quantile(0.95)
print(sales[(sales["Amount"] < p5) | (sales["Amount"] > p95)])

   Transaction_ID  Amount  Verified
0             101     900      True
7             108   20000      True


### Explanation
The 5th and 95th percentiles define the review region. Values outside it are printed. These limits describe rarity in this sample. They do not establish whether a transaction is invalid.

## 6. Winsorization

### Definition
Winsorization replaces values beyond lower and upper cutoffs with cutoff values. It keeps the same number of rows. This can reduce the influence of extreme observations. It also changes the original measurements and should be documented.

### Business Example
A business caps unusually high spending values in a modeling copy of its data. Original transaction amounts remain available for auditing. The capped version helps reduce sensitivity to extremes. Genuine purchases are not erased from the source records.

### AI/ML Use
Winsorization can stabilize models sensitive to large values. It may be unsuitable when extreme magnitudes carry the prediction signal. Choose thresholds using training data only. Apply those same thresholds to validation and test data.

### Python Example

In [13]:
lower = sales["Amount"].quantile(0.05)
upper = sales["Amount"].quantile(0.95)
sales["Winsorized"] = sales["Amount"].clip(lower, upper)
print(sales[["Amount", "Winsorized"]])

   Amount  Winsorized
0     900         935
1    1000        1000
2    1100        1100
3    1200        1200
4    1300        1300
5    1400        1400
6    1500        1500
7   20000       13525


C:\Users\darshan\AppData\Local\Temp\ipykernel_21044\3764688662.py:3: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sales["Winsorized"] = sales["Amount"].clip(lower, upper)


### Explanation
The code calculates lower and upper percentile limits. The clip operation replaces values beyond those limits. Original amounts remain unchanged in their own column. Compare both columns before deciding to use the transformed feature.

## 7. Capping

### Definition
Capping sets a maximum permitted value for a feature. Values above the cap are replaced with the cap. The cap may come from domain knowledge or a statistical rule. Capping reduces numerical influence without removing rows.

### Business Example
An expense model caps unusually high amounts at a chosen modeling limit. The real amount is still stored separately. This prevents a few large values from dominating certain models. Finance staff must approve whether the cap is appropriate.

### AI/ML Use
Capping can help distance-based and linear models. It may hide meaningful differences among large observations. Choose a cap that matches the business objective. Never silently overwrite the only copy of the raw data.

### Python Example

In [15]:
sales["Capped"] = sales["Amount"].clip(upper=5000)
print(sales[["Amount", "Capped"]].tail(2))

   Amount  Capped
6    1500    1500
7   20000    5000


### Explanation
Amounts above ₹5,000 become ₹5,000 in the new column. Smaller amounts are unchanged. The source column preserves actual transaction values. Capping is a modeling choice, not a correction of verified transactions.

## 8. Flooring

### Definition
Flooring sets a minimum allowed value for a feature. Values below the floor are replaced with that minimum. It can be based on physical constraints or a modeling decision. Flooring should not be used to hide data-entry errors.

### Business Example
A sensor reports slightly negative readings even though the true quantity cannot be below zero. Engineers may floor small measurement noise to zero after verification. A large negative reading may instead indicate equipment failure. The cases should be treated differently.

### AI/ML Use
Flooring can enforce known lower bounds in features. It can prevent impossible inputs from reaching a model. But it may conceal a recurring data-quality problem. Check the reason for negative values before applying it.

### Python Example

In [17]:
sensor = pd.Series([-0.2, 0.5, 1.0, 2.0])
print("Floored:", sensor.clip(lower=0).tolist())

Floored: [0.0, 0.5, 1.0, 2.0]


### Explanation
The code replaces negative readings with zero. Positive readings remain unchanged. This is appropriate only if zero is a meaningful lower bound. Investigate the sensor before making this a permanent rule.

## 9. Transformation

### Definition
A transformation changes the scale or distribution of a numerical feature. Log transformation can compress large positive values. It does not delete observations. The chosen transformation must be valid for the feature values.

### Business Example
Customer spending may range from hundreds to tens of thousands. A logarithmic scale makes the gap less extreme. The business still retains the original purchase amounts. Analysts can compare results on both scales.

### AI/ML Use
Log transformations can help models when a feature is strongly right-skewed. The `log1p` function supports zero and positive values. Negative values need separate handling. Transformations should be selected using training data and model needs.

### Python Example

In [19]:
sales["Log_Amount"] = np.log1p(sales["Amount"])
print(sales[["Amount", "Log_Amount"]].tail(3))

   Amount  Log_Amount
5    1400    7.244942
6    1500    7.313887
7   20000    9.903538


### Explanation
The new column contains the natural log of one plus each amount. Very large values are compressed more than small ones. No row is removed. The original column is kept for interpretation and reporting.

## 10. Removing Outliers

### Definition
Removing outliers deletes selected records from the analysis dataset. It can be justified when records are confirmed errors or irrelevant to the task. Removing genuine extreme cases may bias results. The decision should be documented and reproducible.

### Business Example
An employee age of −5 is an invalid entry. If the correct age cannot be recovered, the row may be excluded from a specific analysis. A verified large purchase is different. It should not be removed merely because it is rare.

### AI/ML Use
Removing confirmed errors can improve model quality. Removing rare but valid cases can hurt performance on unusual real-world inputs. Compare model results before and after removal. Keep an audit trail of excluded records.

### Python Example

In [21]:
employees = pd.DataFrame({"Age": [24, 31, -5, 45]})
clean_employees = employees[employees["Age"] >= 0]
print(clean_employees)

   Age
0   24
1   31
3   45


### Explanation
The code excludes the impossible negative age. This removal follows a factual validity rule, not just an outlier threshold. Real extreme ages within a valid range require separate review. Preserve the raw dataset for auditing.

## 11. Retaining Outliers

### Definition
Retaining outliers means keeping unusual observations in the dataset. This is appropriate when they are verified and relevant. Rare cases may be central to the business problem. Models should be evaluated on realistic data that includes such cases.

### Business Example
A bank may see a rare, unusually large transfer. It could be a valid corporate payment or a fraud signal. Deleting it would remove information that matters. Investigators verify it and preserve the transaction record.

### AI/ML Use
Fraud detection and anomaly detection often depend on unusual examples. Tree-based models may be more tolerant of extreme numeric values than some alternatives. Retaining a value does not mean ignoring its effect. Test model performance and consider robust approaches.

### Python Example

In [23]:
verified_sales = sales[sales["Verified"]]
print("Rows retained:", len(verified_sales))
print("Largest retained amount:", verified_sales["Amount"].max())

Rows retained: 8
Largest retained amount: 20000


### Explanation
All verified transactions remain in the dataset. The large purchase is preserved. This demonstrates that statistical rarity is not enough for deletion. Model design should account for legitimate extreme cases.